# Advanced 12: Modern Reranking

`05-reranking` introduced the idea: retrieve broadly with vectors, then re-score
the survivors with a slower model that reads the query and the document *together*.
It used `ms-marco-MiniLM-L-6-v2`, which is from 2021.

This notebook asks two questions that notebook could not:

1. Do newer cross-encoders actually do better on this corpus?
2. Can the LLM you already have act as the reranker, and what does it cost?

The second question matters most for a CPU-friendly project. A cross-encoder is
another model to download and another dependency; `llama3.2:3b` is already here.

**The thing being measured is latency as much as quality.** A reranker that adds
400ms to every query is a different product decision from one that adds 4 seconds,
even if their NDCG is identical. On CPU that difference is the whole story.

In [ ]:
# --- ragkit: shared utilities ---
from ragkit import config, db
from ragkit.embed import embed_texts, embed_one
from ragkit.metrics import ndcg_at_k, precision_at_k
from ragkit.retrieval import cosine_similarity
from ragkit.rerank import DEFAULT_CROSS_ENCODER, cross_encoder_rerank, llm_rerank

import time

print(config.describe())
print(f"baseline cross-encoder: {DEFAULT_CROSS_ENCODER}")
conn = db.connect()

## A corpus with near-misses

Reranking only helps when vector search returns *plausible but wrong* results. If
the top hit is already correct there is nothing to fix, and a reranker is pure
cost.

So this corpus is built adversarially: several documents share vocabulary with
each question while only one actually answers it. That is the situation reranking
exists for, and it is why a benchmark on easy questions will tell you reranking
does nothing.

In [ ]:
DOCS = [
    "Python's list.sort() sorts in place and returns None.",
    "The sorted() builtin returns a new sorted list and leaves the original alone.",
    "Sorting algorithms include quicksort, mergesort, and heapsort.",
    "Python lists are dynamic arrays with amortised O(1) append.",
    "To sort a dictionary by value, pass a key function to sorted().",
    "Timsort, Python's sorting algorithm, is stable and adaptive.",
    "A stable sort preserves the relative order of equal elements.",
    "list.reverse() reverses a list in place, like list.sort().",
]

QUESTIONS = {
    "how do I sort without modifying the original list": [1],
    "is Python's sort stable":                            [5, 6],
    "sort a dict by its values":                          [4],
}

doc_vectors = embed_texts(DOCS)
print(f"{len(DOCS)} documents, {len(QUESTIONS)} questions, dim={len(doc_vectors[0])}")

## Baseline: vectors alone

Retrieve the top 5 by cosine similarity. These are the candidates every reranker
below will be given — same input, so the comparison is fair.

In [ ]:
TOP_N_RETRIEVE = 5
TOP_K_FINAL = 3


def retrieve_candidates(question, n=TOP_N_RETRIEVE):
    qv = embed_one(question)
    scored = sorted(
        ({"index": i, "chunk_text": DOCS[i], "similarity": cosine_similarity(qv, doc_vectors[i])}
         for i in range(len(DOCS))),
        key=lambda d: d["similarity"], reverse=True,
    )
    return scored[:n]


for question, relevant in QUESTIONS.items():
    cands = retrieve_candidates(question)
    ranked = [c["index"] for c in cands]
    print(f"\n{question!r}")
    print(f"  NDCG@3={ndcg_at_k(ranked, relevant, k=TOP_K_FINAL):.3f}   relevant={relevant}")
    for c in cands[:3]:
        mark = "*" if c["index"] in relevant else " "
        print(f"   {mark} {c['similarity']:.3f}  {c['chunk_text'][:62]}")

## Three strategies, measured together

`no rerank` is the control. Without it you cannot tell whether a reranker helped
or whether the retrieval was already fine.

`ragkit.rerank.cross_encoder_rerank` honours its `model_name` argument — worth
noting, because `05-reranking` declared a `RERANKER_MODEL` constant and then
shadowed it with a hardcoded default, so changing the constant did nothing.

In [ ]:
def measure(name, rerank_fn):
    """Run one strategy across all questions, returning mean NDCG and mean latency."""
    ndcgs, latencies = [], []
    for question, relevant in QUESTIONS.items():
        candidates = retrieve_candidates(question)
        start = time.time()
        try:
            ranked = rerank_fn(question, candidates)
        except ImportError as exc:
            print(f"{name:22s} skipped: {exc}")
            return None
        latencies.append(time.time() - start)
        ndcgs.append(ndcg_at_k([c["index"] for c in ranked], relevant, k=TOP_K_FINAL))
    mean_ndcg = sum(ndcgs) / len(ndcgs)
    mean_ms = 1000 * sum(latencies) / len(latencies)
    print(f"{name:22s} NDCG@3={mean_ndcg:.3f}   {mean_ms:8.0f} ms/query")
    return mean_ndcg, mean_ms


print(f"{'strategy':22s} {'quality':14s}   latency")
print("-" * 56)

results = {}
results["none"] = measure("no rerank (control)", lambda q, c: c[:TOP_K_FINAL])
results["cross_encoder"] = measure(
    "cross-encoder", lambda q, c: cross_encoder_rerank(q, c, top_k=TOP_K_FINAL)
)
results["llm"] = measure(
    "LLM-as-reranker", lambda q, c: llm_rerank(q, c, top_k=TOP_K_FINAL)
)

## Reading the table

Look at the two columns together, not the quality column alone.

**The control is the important row.** If reranking does not beat it, you have
added latency and a dependency for nothing. That is a real outcome and worth
reporting honestly rather than tuning until the technique wins.

**Latency is not a footnote on CPU.** The LLM reranker makes one generation call
per candidate. With 5 candidates and a 3B model on CPU, that is seconds per query
— fine for a batch evaluation job, usually unacceptable for anything interactive.
The cross-encoder scores all pairs in a single forward pass and is typically an
order of magnitude faster.

**A small LLM is a noisy judge.** `llm_rerank` asks for a number 0-10 and parses
the first one it finds, clamping to range and falling back to 0.0. That tolerance
is deliberate: small models add commentary despite instructions. But a model that
returns 7 for everything produces an arbitrary ranking that *looks* like a
ranking. Print the scores before trusting them.

In [ ]:
# Look at the actual scores before believing the ranking.
question = next(iter(QUESTIONS))
candidates = retrieve_candidates(question)
scored = llm_rerank(question, candidates, top_k=len(candidates))

print(f"{question!r}\n")
distinct = {c["rerank_score"] for c in scored}
for c in scored:
    print(f"  score={c['rerank_score']:5.1f}  vec={c['similarity']:.3f}  {c['chunk_text'][:58]}")
print(f"\n{len(distinct)} distinct score(s) across {len(scored)} candidates")
if len(distinct) <= 1:
    print("  -> the judge is not discriminating; this ranking is arbitrary.")

## Trying a different cross-encoder

`cross_encoder_rerank` takes any sentence-transformers cross-encoder. Two worth
comparing against the 2021 baseline:

- `BAAI/bge-reranker-base` — stronger, ~1.1GB
- `mixedbread-ai/mxbai-rerank-xsmall-v1` — close in quality, much smaller

The cell below is left for you to run deliberately, because each one downloads a
model. Uncomment when you are ready to spend the bandwidth.

In [ ]:
CANDIDATE_RERANKERS = [
    DEFAULT_CROSS_ENCODER,                    # the 2021 baseline from notebook 05
    # "BAAI/bge-reranker-base",
    # "mixedbread-ai/mxbai-rerank-xsmall-v1",
]

for model_name in CANDIDATE_RERANKERS:
    measure(model_name.split("/")[-1][:22],
            lambda q, c, m=model_name: cross_encoder_rerank(q, c, top_k=TOP_K_FINAL, model_name=m))

## What to take away

Reranking is a **latency-for-quality trade**, and the exchange rate depends
entirely on your corpus. On a corpus where vector search already returns the right
answer first, the trade is bad at any price.

Decide with three numbers, in this order:

1. Does it beat the control at all?
2. What does it cost per query, on the hardware you will actually deploy on?
3. Does the gain hold on real ground truth, not a hand-built demo set?

`evaluation-lab/03` runs that comparison properly with significance testing.
Record this run so it can be compared there.

In [ ]:
from ragkit.experiment import start_experiment, save_metrics, complete_experiment

metrics = {}
for name, result in results.items():
    if result:
        metrics[f"ndcg@3_{name}"] = result[0]
        metrics[f"latency_ms_{name}"] = result[1]

exp_id = start_experiment(
    conn, experiment_name="modern-reranking",
    embedding_model_alias=config.EMBEDDING_ALIAS,
    config={"top_n_retrieve": TOP_N_RETRIEVE, "top_k_final": TOP_K_FINAL,
            "cross_encoder": DEFAULT_CROSS_ENCODER, "llm": config.LLM_TAG},
    notebook_path="advanced-techniques/12-modern-reranking.ipynb",
    techniques=["cross_encoder_rerank", "llm_rerank"],
)
save_metrics(conn, exp_id, metrics, export_to_file=False)
complete_experiment(conn, exp_id)
print(f"recorded as experiment {exp_id} with {len(metrics)} metrics")

conn.close()